# Exercise 3: Evaluation & Optimization

In this exercise we use NAT's built-in tools to systematically evaluate and optimize the workflow.

**Key tools:**
- `nat eval` - run an evaluation dataset through the workflow automatically
- `--override` - change parameters without editing YAML
- `nat serve` - deploy workflow as an API (production pattern)

## What We're Optimizing

| Parameter | Range | Impact |
|-----------|-------|--------|
| `temperature` | 0.0 - 1.0 | Creativity vs. determinism |
| `max_tokens` | 256 - 4096 | Output length vs. cost |

In [ ]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url=os.getenv("VLLM_BASE_URL"),
    api_key=os.getenv("VLLM_API_KEY"),
)
MODEL = os.getenv("VLLM_MODEL_NAME")

## Step 1: Review the Evaluation Dataset

NAT eval needs a JSON dataset with `question` and `answer` fields.

In [ ]:
with open("eval_dataset.json") as f:
    dataset = json.load(f)

print(f"Evaluation dataset: {len(dataset)} questions\n")
for item in dataset[:3]:
    print(f"  Q: {item['question']}")
    print(f"  A: {item['answer']}")
    print()

## Step 2: Run `nat eval` - Baseline Evaluation

`nat eval` runs all 8 questions through the workflow automatically.
The config includes an `eval` section pointing to our dataset.

In [ ]:
with open("config_eval.yaml") as f:
    print(f.read())

In [ ]:
!nat eval --config_file config_eval.yaml

## Step 3: Parameter Sweep with `--override`

The `--override` flag lets you change any config value without editing YAML.
Let's compare temperature=0.1 (conservative) vs temperature=0.7 (default).

In [ ]:
print("=== Temperature 0.1 (conservative) ===")
!nat eval --config_file config_eval.yaml --override llms.bielik.temperature 0.1

print("\n=== Temperature 0.7 (default) ===")
!nat eval --config_file config_eval.yaml --override llms.bielik.temperature 0.7

## Step 4: Manual Temperature Sweep

For a quick comparison, let's test a single question at different temperatures.

In [ ]:
question = "W którym roku Polska przystąpiła do Unii Europejskiej?"
expected = "2004"

INPUT_RATE = 0.10 / 1_000_000
OUTPUT_RATE = 0.20 / 1_000_000

print(f"Question: {question}")
print(f"Expected: {expected}")
print("=" * 70)

for temp in [0.0, 0.3, 0.5, 0.7, 1.0]:
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=temp,
        max_tokens=128,
    )
    latency = (time.time() - start) * 1000
    answer = response.choices[0].message.content.strip()
    tokens = response.usage.total_tokens
    cost = response.usage.prompt_tokens * INPUT_RATE + response.usage.completion_tokens * OUTPUT_RATE
    correct = expected.lower() in answer.lower()
    
    print(f"\ntemp={temp:.1f} | {latency:.0f}ms | {tokens} tok | ${cost:.6f} | {'OK' if correct else 'MISS'}")
    print(f"  -> {answer[:100]}")

## Step 5: Max Tokens Impact on Cost

`max_tokens` is a cost ceiling per call. Lower it = cheaper, but may truncate answers.

In [ ]:
question = "Czym jest Kopalnia Soli w Wieliczce?"

print(f"Question: {question}\n")
print(f"{'max_tokens':<12} {'Output':>8} {'Cost':>10} {'Answer preview'}")
print("-" * 70)

for max_tok in [64, 128, 256, 512, 1024]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0.5,
        max_tokens=max_tok,
    )
    out_tokens = response.usage.completion_tokens
    cost = response.usage.prompt_tokens * INPUT_RATE + out_tokens * OUTPUT_RATE
    answer = response.choices[0].message.content[:60].replace('\n', ' ')
    print(f"{max_tok:<12} {out_tokens:>8} ${cost:>8.6f}   {answer}...")

print(f"\nA workflow with 7 LLM calls at max_tokens=1024 vs 256: ~4x cost difference.")

## Step 6: Deploy as API with `nat serve` (Demo)

In production, you'd deploy the workflow as a REST API and benchmark it:

```bash
# Terminal 1: Start the server
nat serve --config_file config.yaml --port 8080

# Terminal 2: Test it
curl -X POST http://localhost:8080/generate \
  -H 'Content-Type: application/json' \
  -d '{"input": "Kim był Chopin?"}'

# Terminal 2: Evaluate against the served endpoint
nat eval --config_file config_eval.yaml --endpoint http://localhost:8080/generate
```

This separates deployment from evaluation - the production pattern.

## Feedback

![QR Code](feedback_qr.png)